# BGE-small + FAISS full-corpus baseline (Colab GPU)

Embeds all 511,958 EnterpriseRAG-Bench documents with the pinned `BAAI/bge-small-en-v1.5`
model and scores the development split, like the BM25+ baseline.

1. **Runtime → Change runtime type → T4 GPU.**
2. Run the cells in order. Expect about 10 minutes of setup and about 2 hours of embedding (a T4 took 1 h 50 min).
   The corpus is reused from Drive if it is already there.
3. Work is saved to Google Drive in shards, so after a disconnect rerun all cells and it resumes.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Google Drive for resumable storage

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/rag-dense"  # Corpus Parquet and embedding shards go here.
!mkdir -p {WORK}

## 2. Get the project code


In [ ]:
import os

BRANCH = "hira/dev"
if not os.path.exists("/content/project"):
    !git clone --depth 1 -b {BRANCH} https://github.com/hira-mian/Agentic-Enterprise-RAG-System.git /content/project
%cd /content/project
!git log -1 --format='%h %s'

In [ ]:
!pip install -q -r requirements.txt

## 3. Build the full corpus (skipped if already on Drive)

Clones the pinned upstream repository and converts it to Parquet (about 5–10 minutes).

In [ ]:
CORPUS = f"{WORK}/documents.parquet"
if not os.path.exists(CORPUS):
    !python -m src.data.corpus --clone --output {CORPUS}
else:
    print("Using existing", CORPUS)

## 4. Embed passages on the GPU

Each document is split into 500-token windows with 50-token overlap (about 1.7 million
passages). Completed shards are skipped when this cell is rerun.

`TITLE_EVERY_PASSAGE = True` puts each document's title at the start of every passage
(not just the first) and writes to a separate folder, so the original baseline is kept.
Set it to `False` to reproduce the original baseline.

In [ ]:
TITLE_EVERY_PASSAGE = True
NAME = "dense_titled" if TITLE_EVERY_PASSAGE else "dense"
FLAG = "--title-every-passage" if TITLE_EVERY_PASSAGE else ""
RESULTS = f"{NAME}_full_development"

!python -m src.retrieval.corpus_dense --corpus {CORPUS} --output {WORK}/{NAME} --device cuda --batch-size 256 {FLAG}

## 5. Score the development split

In [ ]:
!python -m src.evaluation.run_retrieval --method dense --index {WORK}/{NAME} --device cuda --split development \
    --questions {WORK}/questions.jsonl --output evaluation/results/{RESULTS}

In [ ]:
import json

def load(name):
    return json.load(open(f"evaluation/results/{name}/summary.json"))

new, old, bm25 = load(RESULTS), load("dense_full_development"), load("bm25_full_development")
print(f"{'type':26} {'scored':>6} {'BGE before':>11} {'BGE after':>10} {'change':>8} {'BM25+':>7}   (Recall@10)")
rows = [("Overall", "overall")] + [(t, t) for t in new["question_types"]]
for label, key in rows:
    pick = (lambda s: s["overall"]) if key == "overall" else (lambda s: s["question_types"][key])
    a, b, c = pick(old)["recall@10"]["mean"], pick(new)["recall@10"]["mean"], pick(bm25)["recall@10"]["mean"]
    if b is None:
        continue
    print(f"{label:26} {pick(new)['recall@10']['scored']:>6} {a:>11.3f} {b:>10.3f} {b - a:>+8.3f} {c:>7.3f}")
print()
for metric in ("recall@5", "recall@20", "ndcg@10"):
    a, b = old["overall"][metric]["mean"], new["overall"][metric]["mean"]
    print(f"{metric:>10}: before {a:.3f} -> after {b:.3f} ({b - a:+.3f})")

## 6. Save the results

This copies the result files to Drive (`rag-dense/results`) and downloads a zip.
Commit the results folder to `evaluation/results/`, or share the Drive folder.

In [ ]:
!mkdir -p {WORK}/results && cp -r evaluation/results/{RESULTS} {WORK}/results/
!cd evaluation/results && zip -qr /content/{RESULTS}.zip {RESULTS}
from google.colab import files
files.download(f"/content/{RESULTS}.zip")